# Lab 2: Select — Picking Relevant Content\n\n**Concept:** Selection is the core of context engineering: choose only the sentences/tokens that matter for answering the question. Methods range from simple keyword matching to TF-IDF similarity to token-level importance scoring.\n\n## Exercises\n- 2a. Compare keyword selection strategies (basic, bigrams, synonym expansion) on the same document (from `naive_regex`).\n- 2b. Use TF-IDF cosine similarity to select the top-N most relevant chunks (from `semantic_chunk`).\n- 2c. Ask an LLM to score token importance, simulating LLMLingua's token-level selection (from `llmlingua`).\n

## Setup\n\n```bash\npip install openai scikit-learn numpy\nexport OPENROUTER_API_KEY='sk-or-v1-...'\n```

In [ ]:
import os, re, json, pickle, glob\nimport numpy as np\nfrom openai import OpenAI\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.metrics.pairwise import cosine_similarity\n\nclient = OpenAI(api_key=os.getenv('OPENROUTER_API_KEY'), base_url='https://openrouter.ai/api/v1')

## Exercises

In [ ]:
import os, re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

client = OpenAI(api_key=os.getenv("OPENROUTER_API_KEY"), base_url="https://openrouter.ai/api/v1")

document = """The company reported strong Q3 earnings. Revenue grew by 15% year over year to $45.2 million.
The expansion of the Frankfurt server farm cost $14.73 million. This was partially offset by a $2.1 million
green energy tax rebate. Employee headcount remained stable at 1,200."""
sentences = re.split(r'(?<=[.!?])\s+', document)
question = "What offset the Frankfurt data center costs?"

# === 2a: Keyword strategies ===
def keywords_for(question, strategy="basic"):
    if strategy == "basic":
        return {w.lower() for w in re.findall(r'\b[a-zA-Z]{3,}\b', question)}
    elif strategy == "expand":
        base = {w.lower() for w in re.findall(r'\b[a-zA-Z]{3,}\b', question)}
        syns = {"cost": {"expense", "capex"}, "offset": {"rebate", "credit"}}
        for w in list(base):
            if w in syns:
                base |= syns[w]
        return base

print("=== 2a: Keyword Strategy Comparison ===")
for strat in ["basic", "expand"]:
    kw = keywords_for(question, strat)
    sel = [s for s in sentences if any(k in s.lower() for k in kw)]
    print(f"  {strat}: {len(sel)} sentences matched (keywords: {kw})")

# === 2b: TF-IDF top-N selection ===
vect = TfidfVectorizer(ngram_range=(1,2))
vect.fit([question] + sentences)
scores = cosine_similarity(vect.transform([question]), vect.transform(sentences)).flatten()
best = np.argsort(scores)[::-1][:2]
print("\n=== 2b: TF-IDF Top-2 Chunks ===")
for i in best:
    print(f"  [{scores[i]:.3f}] {sentences[i]}")

# === 2c: Token importance scoring (simulating LLMLingua) ===
resp = client.chat.completions.create(
    model="meta-llama/llama-3.3-70b-instruct:free",
    messages=[{"role": "user", "content":
        f"Document: {document}\n\nQuestion: {question}\n\n"
        f"List each word's importance as HIGH/MEDIUM/LOW. "
        f"Format: word -> level"}],
    temperature=0,
)
print("\n=== 2c: Token Importance ===")
print(resp.choices[0].message.content[:200])

## Summary\n\nSelection is the core of context engineering: choose only the sentences/tokens that matter for answering the question. Methods range from simple keyword matching to TF-IDF similarity to token-level importance scoring.